# ADMET Multi-Task Model – Full Documentation


---

Model służy do przewidywania wielu właściwości ADMET jednocześnie (multi-task learning) na podstawie SMILES. Każda próbka reprezentuje jedną cząsteczkę i jeden task, ale model uczy się wszystkich tasków wspólnie poprzez maskowanie i współdzieloną reprezentację.

---

Pipeline zaczyna się od wczytania danych z pliku parquet. Każdy rekord zawiera SMILES, label, task oraz zestaw feature’ów: ~200 cech RDKit (2D deskryptory), 4 cechy kwantowe (dipole, HOMO-LUMO gap, liczba elektronów, energia) oraz 4-elementową maskę wskazującą, czy dane QC istnieją. Dodatkowo każda próbka ma przypisany split (train/valid/test).

---

Dane są przekształcane do formy multi-task: dla T tasków tworzony jest wektor y ∈ R^T oraz maska mask ∈ {0,1}^T. Tylko jeden element y jest ustawiony (dla danego tasku), reszta to zera, a maska wskazuje które zadanie jest aktywne. Dzięki temu model uczy się wielu tasków jednocześnie bez mieszania labeli.


---

Feature engineering polega na połączeniu trzech źródeł informacji: RDKit (globalne właściwości molekuły), quantum features (informacje fizyczne) oraz maska QC. Dane są czyszczone poprzez zamianę NaN i inf na 0 oraz clipping do zakresu [-1e6, 1e6], co zapobiega niestabilności numerycznej.

---

Następnie wykonywana jest standaryzacja (normalizacja) tylko na zbiorze treningowym. RDKit i QC są normalizowane osobno: x = (x - mean) / std, gdzie std < 1e-6 jest zastępowane przez 1. Maski QC nie są normalizowane. Po normalizacji stosowane jest maskowanie QC: qc = qc * qc_mask, co oznacza, że brakujące wartości są zerowane, ale model nadal wie, że ich nie było dzięki masce.


---

Podział danych wykonywany jest przez scaffold split (Murcko scaffold), czyli molekuły o podobnym rdzeniu chemicznym trafiają do jednego splitu. Zapewnia to brak leakage między train/val/test i bardziej realistyczną ewaluację.


---

Każdy SMILES jest konwertowany do grafu: atomy są wierzchołkami, wiązania krawędziami. Graf jest skierowany (każde wiązanie występuje jako i→j i j→i). Dla atomów budowane są feature’y obejmujące typ atomu, stopień, ładunek, hybrydyzację, aromaticity, ring oraz liczbę wodoru. Dla wiązań tworzone są cechy opisujące typ wiązania, sprzężenie, aromaticity, ring oraz stereo.


---


Encoder to D-MPNN (Directed Message Passing Neural Network), gdzie wiadomości propagują się po wiązaniach zamiast po atomach. Inicjalnie dla każdej krawędzi tworzony jest wektor wiadomości na podstawie atomu źródłowego i cech wiązania. Następnie przez kilka iteracji (depth) wiadomości są aktualizowane poprzez agregację sąsiadów z wykluczeniem poprzedniej wiadomości (nei - m), co zapobiega przepływowi informacji do samej siebie. Po propagacji wiadomości są agregowane do atomów, a embedding atomu jest aktualizowany z użyciem residual connection.

---


Embedding całej molekuły powstaje przez sumowanie embeddingów atomów (sum pooling), co daje wektor z reprezentujący strukturę molekuły.


---

Ten embedding jest łączony z feature’ami tabularnymi (RDKit + QC + maska), co daje końcowy wektor wejściowy o wymiarze 508. Ten wektor trafia do wspólnej sieci MLP (shared encoder), która ma strukturę 508 → 512 → 512 → 256 z aktywacją ReLU i dropoutem. Celem tej części jest nauczenie wspólnej reprezentacji dla wszystkich tasków.


---

Na końcu model posiada osobne głowy (heads) dla każdego tasku — każda to warstwa liniowa 256 → 1. Dzięki temu model współdzieli reprezentację, ale decyzja dla każdego tasku jest niezależna.

---


Loss funkcja opiera się na Binary Cross Entropy z maskowaniem. Najpierw liczony jest loss dla każdego tasku osobno, tylko dla aktywnych próbek. Następnie wprowadzane jest ważenie tasków. Dla każdego tasku liczy się liczba próbek n_t oraz udział r_t = n_t / sum(n). Model uczy parametr log_beta_t, który po transformacji softplus daje β_t > 0. Waga tasku to w_t = r_t^β_t.


---

Końcowy loss to suma ważona: L = sum_t w_t * L_t. Intuicyjnie oznacza to, że taski z dużą liczbą próbek są automatycznie downweightowane, a małe taski dostają większy wpływ. Dodatkowo stosowany jest pos_weight w BCE, który kompensuje imbalance klas (neg/pos).


---

Backpropagation przebiega standardowo przez wszystkie komponenty: heady → shared MLP → concatenation → D-MPNN → atom/bond features. W celu stabilizacji treningu stosowane są: mixed precision (AMP), gradient clipping (norma ≤ 5) oraz ograniczenie wartości log_beta do zakresu [-3, 3].


---

Cały model łączy trzy kluczowe źródła informacji: strukturę (GNN), właściwości fizykochemiczne (RDKit) oraz informacje kwantowe (QC). Multi-task learning umożliwia transfer wiedzy między zadaniami, a dynamiczne ważenie lossu zapobiega dominacji dużych datasetów. Dzięki temu model osiąga lepszą generalizację niż podejście single-task.



In [ ]:
!pip install chemprop==2.2.2
!pip install torch>=2.0


import chemprop
print(chemprop.__version__)
!pip check

2.2.2
shap 0.51.0 has requirement numpy>=2, but you have numpy 1.26.4.
rasterio 1.5.0 has requirement numpy>=2, but you have numpy 1.26.4.
tobler 0.13.0 has requirement numpy>=2.0, but you have numpy 1.26.4.
jaxlib 0.7.2 has requirement numpy>=2.0, but you have numpy 1.26.4.
jax 0.7.2 has requirement numpy>=2.0, but you have numpy 1.26.4.
opencv-python 4.13.0.92 has requirement numpy>=2; python_version >= "3.9", but you have numpy 1.26.4.
xarray-einstats 0.10.0 has requirement numpy>=2.0, but you have numpy 1.26.4.
opencv-python-headless 4.13.0.92 has requirement numpy>=2; python_version >= "3.9", but you have numpy 1.26.4.
pytensor 2.38.2 has requirement numpy>=2.0, but you have numpy 1.26.4.
opencv-contrib-python 4.13.0.92 has requirement numpy>=2; python_version >= "3.9", but you have numpy 1.26.4.


In [ ]:
!pip install rdkit
!pip install fastparquet
!pip install pyarrow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.7/36.7 MB 76.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 86.3 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
%cd /content

/content


In [ ]:
import pandas as pd
import numpy as np

# ===== LOAD =====
df = pd.read_parquet("dataset_raw.parquet")

tasks = sorted(df["task"].unique())
task_to_idx = {t:i for i,t in enumerate(tasks)}

qc_cols = ["dipole","homo_lumo","electrons","energy"]
qc_mask_cols = ["mask_dipole","mask_homo_lumo","mask_electrons","mask_energy"]
meta_cols = ["smiles","label","task","split","success"]

rd_cols = [c for c in df.columns if c not in meta_cols + qc_cols + qc_mask_cols]

X_list, y_list, m_list, smiles_list, split_list = [], [], [], [], []

# ===== BUILD =====
for _, r in df.iterrows():
    # RDKit
    rd = r[rd_cols].values.astype(np.float64)
    rd = np.nan_to_num(rd, nan=0.0, posinf=0.0, neginf=0.0)
    rd = np.clip(rd, -1e6, 1e6).astype(np.float32)

    # QC
    qc = r[qc_cols].values.astype(np.float64)
    qc = np.nan_to_num(qc, nan=0.0, posinf=0.0, neginf=0.0)
    qc = np.clip(qc, -1e6, 1e6).astype(np.float32)

    qc_mask = r[qc_mask_cols].values.astype(np.float32)

    # concat
    x = np.concatenate([rd, qc, qc_mask]).astype(np.float32)

    # multitask target
    y = np.zeros(len(tasks), dtype=np.float32)
    m = np.zeros(len(tasks), dtype=np.float32)

    t = task_to_idx[r["task"]]
    y[t] = r["label"]
    m[t] = 1

    X_list.append(x)
    y_list.append(y)
    m_list.append(m)
    smiles_list.append(r["smiles"])
    split_list.append(r["split"])


smiles_list = np.array(smiles_list)
X = np.stack(X_list)
y = np.stack(y_list)
mask = np.stack(m_list)

# ===== SPLIT =====
split_arr = np.array(split_list)

train_idx = np.where(split_arr=="train")[0]
val_idx   = np.where(split_arr=="valid")[0]
test_idx  = np.where(split_arr=="test")[0]

# ===== FEATURE SPLIT =====
rd = X[:, :200]
qc = X[:, 200:204]
qc_mask = X[:, 204:208]

# ===== NORMALIZATION (TYLKO RD + QC) =====

# RD
mean_rd = np.nanmean(rd[train_idx], axis=0)
std_rd  = np.nanstd(rd[train_idx], axis=0)
mean_rd = np.nan_to_num(mean_rd, nan=0.0)
std_rd  = np.nan_to_num(std_rd, nan=1.0)
std_rd[std_rd < 1e-6] = 1.0
rd = (rd - mean_rd) / std_rd

# QC
mean_qc = np.nanmean(qc[train_idx], axis=0)
std_qc  = np.nanstd(qc[train_idx], axis=0)
mean_qc = np.nan_to_num(mean_qc, nan=0.0)
std_qc  = np.nan_to_num(std_qc, nan=1.0)
std_qc[std_qc < 1e-6] = 1.0
qc = (qc - mean_qc) / std_qc

# 🔥 KLUCZOWE — maskowanie QC
qc = qc * qc_mask

# ===== FINAL X =====
X = np.concatenate([rd, qc, qc_mask], axis=1)
X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)

# ===== DEBUG =====
print("\n=== DATA CHECK ===")
print("Shape:", X.shape)
print("NaN:", np.isnan(X).sum())
print("INF:", np.isinf(X).sum())
print("MIN:", X.min(), "MAX:", X.max())
print("MEAN:", X.mean(), "STD:", X.std())

print("\n=== SPLITS ===")
print("train:", len(train_idx), "val:", len(val_idx), "test:", len(test_idx))

print("\n=== TARGET ===")
print("y unique:", np.unique(y))
print("mask per task:", mask.sum(0))

# zero variance
zero_std = np.where(np.std(X, axis=0) < 1e-8)[0]
print("zero-std features:", len(zero_std))

# QC mask sanity (TERAZ POPRAWNE)
print("\nQC mask sum:", qc_mask.sum(axis=0))

# QC real values check
print("QC values mean:", qc.mean(), "std:", qc.std())

# RD sanity
print("RD mean:", rd.mean(), "std:", rd.std())


print("\nOK — DATA READY (ZGODNE Z PAPEREM)")


=== DATA CHECK ===
Shape: (65127, 208)
NaN: 0
INF: 0
MIN: -43.09469 MAX: 128.80121
MEAN: 0.019848928 STD: 0.9957309

=== SPLITS ===
train: 45617 val: 6508 test: 13002

=== TARGET ===
y unique: [0. 1.]
mask per task: [ 7255.  1975.   640.   666. 12092.   664. 13130.   667. 12328.   475.
 13445.   578.  1212.]
zero-std features: 2

QC mask sum: [63878. 63878. 63878. 63878.]
QC values mean: 0.018271917 std: 0.94492084
RD mean: 0.00066103047 std: 0.9970297

OK — DATA READY (ZGODNE Z PAPEREM)


In [ ]:
# asserty
assert not np.isnan(X).any()
assert not np.isinf(X).any()
assert X.shape[1] == 208
assert y.shape[1] == len(tasks)

print("\nOK — DATA READY")


OK — DATA READY


In [ ]:
import numpy as np
import pandas as pd
from collections import Counter, defaultdict

# =========================================
# 🔥 FULL SPLIT / LEAKAGE / DATA DEBUG
# =========================================
def debug_dataset(smiles_list, split_arr, y, mask):

    smiles = np.array(smiles_list)
    split  = np.array(split_arr)

    train_idx = np.where(split=="train")[0]
    val_idx   = np.where(split=="valid")[0]
    test_idx  = np.where(split=="test")[0]

    train_smiles = smiles[train_idx]
    val_smiles   = smiles[val_idx]
    test_smiles  = smiles[test_idx]

    print("\n==============================")
    print("🔍 BASIC SPLIT STATS")
    print("==============================")
    print("Total samples:", len(smiles))
    print("Unique SMILES:", len(set(smiles)))
    print("Train:", len(train_idx), "Val:", len(val_idx), "Test:", len(test_idx))

    # =========================================
    # 🔴 LEAKAGE CHECK
    # =========================================
    print("\n==============================")
    print("🚨 LEAKAGE CHECK")
    print("==============================")

    train_set = set(train_smiles)
    val_set   = set(val_smiles)
    test_set  = set(test_smiles)

    inter_tv = train_set & val_set
    inter_tt = train_set & test_set
    inter_vt = val_set & test_set

    print("Train ∩ Val:", len(inter_tv))
    print("Train ∩ Test:", len(inter_tt))
    print("Val   ∩ Test:", len(inter_vt))

    if len(inter_tv) > 0 or len(inter_tt) > 0 or len(inter_vt) > 0:
        print("❌ DATA LEAKAGE DETECTED")
    else:
        print("✔ No leakage")

    # =========================================
    # 🔁 DUPLICATES ANALYSIS
    # =========================================
    print("\n==============================")
    print("🔁 DUPLICATES")
    print("==============================")

    counts = Counter(smiles)
    dup = [s for s,c in counts.items() if c > 1]

    print("Total duplicates (SMILES appearing >1):", len(dup))

    # how duplicates distributed across splits
    leak_multi = 0
    for s in dup:
        idx = np.where(smiles == s)[0]
        splits = set(split[idx])
        if len(splits) > 1:
            leak_multi += 1

    print("Duplicates across splits:", leak_multi)

    # =========================================
    # 📊 TASK DISTRIBUTION
    # =========================================
    print("\n==============================")
    print("📊 TASK STATS")
    print("==============================")

    n_tasks = y.shape[1]

    for t in range(n_tasks):
        m = mask[:, t] == 1
        y_t = y[m, t]

        if len(y_t) == 0:
            continue

        pos = y_t.sum()
        neg = len(y_t) - pos

        print(f"Task {t:02d} | samples={len(y_t):5d} | pos={int(pos):5d} | neg={int(neg):5d} | ratio={pos/(len(y_t)+1e-8):.3f}")

    # =========================================
    # ⚖️ CLASS IMBALANCE PER SPLIT
    # =========================================
    print("\n==============================")
    print("⚖️ SPLIT CLASS BALANCE")
    print("==============================")

    def split_stats(name, idx):
        print(f"\n{name}")
        for t in range(n_tasks):
            m = mask[idx, t] == 1
            y_t = y[idx, t][m]

            if len(y_t) < 10:
                continue

            pos = y_t.sum()
            print(f"Task {t:02d} | n={len(y_t):5d} | pos%={pos/(len(y_t)+1e-8):.3f}")

    split_stats("TRAIN", train_idx)
    split_stats("VAL", val_idx)
    split_stats("TEST", test_idx)

    # =========================================
    # 🧪 MOLECULE FREQUENCY
    # =========================================
    print("\n==============================")
    print("🧪 MOLECULE FREQUENCY")
    print("==============================")

    freq = list(counts.values())
    print("Mean freq:", np.mean(freq))
    print("Max freq:", np.max(freq))

    hist = Counter(freq)
    print("Freq histogram (value=count):")
    for k in sorted(hist.keys())[:10]:
        print(f"{k}: {hist[k]}")

    # =========================================
    # 🔍 SAME MOLECULE DIFFERENT LABELS
    # =========================================
    print("\n==============================")
    print("🔍 SAME SMILES DIFFERENT LABELS")
    print("==============================")

    smiles_to_labels = defaultdict(list)

    for i in range(len(smiles)):
        smiles_to_labels[smiles[i]].append(y[i])

    inconsistent = 0
    for s, ys in smiles_to_labels.items():
        arr = np.stack(ys)
        if np.any(np.std(arr, axis=0) > 0):
            inconsistent += 1

    print("SMILES with different labels across rows:", inconsistent)

    # =========================================
    # 📉 HARD TASKS
    # =========================================
    print("\n==============================")
    print("📉 HARD TASKS (low samples)")
    print("==============================")

    task_counts = mask.sum(0)
    for t, c in enumerate(task_counts):
        if c < 1000:
            print(f"Task {t:02d} LOW DATA: {int(c)}")

    print("\n==============================")
    print("✅ DEBUG DONE")
    print("==============================")

In [ ]:
from rdkit import Chem
from rdkit.Chem.Scaffolds import MurckoScaffold

def scaffold(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    return MurckoScaffold.MurckoScaffoldSmiles(mol=mol)

def debug_scaffold_leakage(smiles_list, train_idx, val_idx, test_idx):

    smiles = np.array(smiles_list)

    train_scaf = set(scaffold(s) for s in smiles[train_idx])
    val_scaf   = set(scaffold(s) for s in smiles[val_idx])
    test_scaf  = set(scaffold(s) for s in smiles[test_idx])

    print("\n=== SCAFFOLD LEAKAGE ===")
    print("Train ∩ Val:", len(train_scaf & val_scaf))
    print("Train ∩ Test:", len(train_scaf & test_scaf))
    print("Val ∩ Test:", len(val_scaf & test_scaf))

In [ ]:
debug_dataset(smiles_list, split_arr, y, mask)


🔍 BASIC SPLIT STATS
Total samples: 65127
Unique SMILES: 40000
Train: 45617 Val: 6508 Test: 13002

🚨 LEAKAGE CHECK
Train ∩ Val: 3116
Train ∩ Test: 5815
Val   ∩ Test: 1230
❌ DATA LEAKAGE DETECTED

🔁 DUPLICATES
Total duplicates (SMILES appearing >1): 15080
Duplicates across splits: 8565

📊 TASK STATS
Task 00 | samples= 7255 | pos= 3951 | neg= 3304 | ratio=0.545
Task 01 | samples= 1975 | pos= 1504 | neg=  471 | ratio=0.762
Task 02 | samples=  640 | pos=  492 | neg=  148 | ratio=0.769
Task 03 | samples=  666 | pos=  141 | neg=  525 | ratio=0.212
Task 04 | samples=12092 | pos= 4045 | neg= 8047 | ratio=0.335
Task 05 | samples=  664 | pos=  191 | neg=  473 | ratio=0.288
Task 06 | samples=13130 | pos= 2514 | neg=10616 | ratio=0.191
Task 07 | samples=  667 | pos=  354 | neg=  313 | ratio=0.531
Task 08 | samples=12328 | pos= 5110 | neg= 7218 | ratio=0.415
Task 09 | samples=  475 | pos=  236 | neg=  239 | ratio=0.497
Task 10 | samples=13445 | pos= 6718 | neg= 6727 | ratio=0.500
Task 11 | samples=

In [ ]:
from rdkit import Chem
from rdkit.Chem.Scaffolds import MurckoScaffold
import numpy as np
from collections import defaultdict

def scaffold(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    return MurckoScaffold.MurckoScaffoldSmiles(mol=mol)

def scaffold_split(smiles_list, frac=(0.7,0.1,0.2), seed=42):
    groups = defaultdict(list)

    for i, s in enumerate(smiles_list):
        groups[scaffold(s)].append(i)

    groups = list(groups.values())
    rng = np.random.RandomState(seed)
    rng.shuffle(groups)

    n = len(smiles_list)
    n_train = int(frac[0]*n)
    n_val   = int(frac[1]*n)

    train, val, test = [], [], []

    for g in groups:
        if len(train) + len(g) <= n_train:
            train += g
        elif len(val) + len(g) <= n_val:
            val += g
        else:
            test += g

    return np.array(train), np.array(val), np.array(test)

In [ ]:


train_idx, val_idx, test_idx = scaffold_split(smiles_list)
split_arr = np.array(["train"] * len(smiles_list))
split_arr[val_idx] = "valid"
split_arr[test_idx] = "test"

[10:24:04] WARNING: not removing hydrogen atom without neighbors
[10:24:04] WARNING: not removing hydrogen atom without neighbors
[10:24:04] WARNING: not removing hydrogen atom without neighbors
[10:24:04] WARNING: not removing hydrogen atom without neighbors
[10:24:04] WARNING: not removing hydrogen atom without neighbors
[10:24:04] WARNING: not removing hydrogen atom without neighbors
[10:24:04] WARNING: not removing hydrogen atom without neighbors
[10:24:04] WARNING: not removing hydrogen atom without neighbors
[10:24:04] WARNING: not removing hydrogen atom without neighbors
[10:24:04] WARNING: not removing hydrogen atom without neighbors
[10:24:04] WARNING: not removing hydrogen atom without neighbors
[10:24:04] WARNING: not removing hydrogen atom without neighbors
[10:24:04] WARNING: not removing hydrogen atom without neighbors
[10:24:04] WARNING: not removing hydrogen atom without neighbors
[10:24:04] WARNING: not removing hydrogen atom without neighbors
[10:24:04] WARNING: not r

In [ ]:

debug_dataset(smiles_list, split_arr, y, mask)
debug_scaffold_leakage(smiles_list, train_idx, val_idx, test_idx)


🔍 BASIC SPLIT STATS
Total samples: 65127
Unique SMILES: 40000
Train: 45588 Val: 6512 Test: 13027

🚨 LEAKAGE CHECK
Train ∩ Val: 0
Train ∩ Test: 0
Val   ∩ Test: 0
✔ No leakage

🔁 DUPLICATES
Total duplicates (SMILES appearing >1): 15080
Duplicates across splits: 0

📊 TASK STATS
Task 00 | samples= 7255 | pos= 3951 | neg= 3304 | ratio=0.545
Task 01 | samples= 1975 | pos= 1504 | neg=  471 | ratio=0.762
Task 02 | samples=  640 | pos=  492 | neg=  148 | ratio=0.769
Task 03 | samples=  666 | pos=  141 | neg=  525 | ratio=0.212
Task 04 | samples=12092 | pos= 4045 | neg= 8047 | ratio=0.335
Task 05 | samples=  664 | pos=  191 | neg=  473 | ratio=0.288
Task 06 | samples=13130 | pos= 2514 | neg=10616 | ratio=0.191
Task 07 | samples=  667 | pos=  354 | neg=  313 | ratio=0.531
Task 08 | samples=12328 | pos= 5110 | neg= 7218 | ratio=0.415
Task 09 | samples=  475 | pos=  236 | neg=  239 | ratio=0.497
Task 10 | samples=13445 | pos= 6718 | neg= 6727 | ratio=0.500
Task 11 | samples=  578 | pos=  500 | neg

[10:24:51] WARNING: not removing hydrogen atom without neighbors
[10:24:51] WARNING: not removing hydrogen atom without neighbors
[10:24:52] WARNING: not removing hydrogen atom without neighbors
[10:24:53] WARNING: not removing hydrogen atom without neighbors
[10:24:53] WARNING: not removing hydrogen atom without neighbors
[10:24:53] WARNING: not removing hydrogen atom without neighbors
[10:24:54] WARNING: not removing hydrogen atom without neighbors
[10:24:54] WARNING: not removing hydrogen atom without neighbors
[10:24:54] WARNING: not removing hydrogen atom without neighbors
[10:24:57] WARNING: not removing hydrogen atom without neighbors
[10:24:58] WARNING: not removing hydrogen atom without neighbors
[10:24:58] WARNING: not removing hydrogen atom without neighbors
[10:24:58] WARNING: not removing hydrogen atom without neighbors
[10:24:59] WARNING: not removing hydrogen atom without neighbors
[10:25:01] WARNING: not removing hydrogen atom without neighbors
[10:25:02] WARNING: not r


=== SCAFFOLD LEAKAGE ===
Train ∩ Val: 0
Train ∩ Test: 0
Val ∩ Test: 0


In [ ]:
# =========================
# HYPERPARAMS
# =========================
LR = 2e-4
WEIGHT_DECAY = 1e-5
EPOCHS = 60
PATIENCE = 8

BATCH_SIZE = 1024

HIDDEN = 300
DEPTH = 4

FOCAL_GAMMA = 2.0
BETA_SCALE = 2.0

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from rdkit import Chem
from torch.utils.data import Dataset


# =========================
# UTILS (FEATURES)
# =========================



def collate_fn(batch):
    atoms, edges, bonds, revs, Xs, ys, masks = zip(*batch)

    xs, es, bs, rs, batch_idx = [], [], [], [], []
    offset = 0
    e_offset = 0

    for i in range(len(atoms)):
        x = atoms[i]
        e = edges[i]
        b = bonds[i]
        r = revs[i]

        xs.append(x)
        es.append(e + offset)
        bs.append(b)
        rs.append(r + e_offset)

        batch_idx.append(torch.full((x.size(0),), i, dtype=torch.long))

        offset += x.size(0)
        e_offset += e.size(1)

    return (
        torch.cat(xs),
        torch.cat(es, 1),
        torch.cat(bs),
        torch.cat(rs),
        torch.cat(batch_idx),
        torch.stack(Xs),
        torch.stack(ys),
        torch.stack(masks)
    )



class MolDataset(Dataset):
    def __init__(self, smiles, X, y, mask, indices):
        self.smiles = [smiles[i] for i in indices]
        self.X = X[indices]
        self.y = y[indices]
        self.mask = mask[indices]

        # 🔥 cache grafów
        self.graphs = [mol_to_graph(s) for s in self.smiles]

    def __len__(self):
        return len(self.X)

    def __getitem__(self, i):
        atom, edge, bond, rev = self.graphs[i]

        return (
            atom,
            edge,
            bond,
            rev,
            torch.tensor(self.X[i], dtype=torch.float32),
            torch.tensor(self.y[i], dtype=torch.float32),
            torch.tensor(self.mask[i], dtype=torch.float32)
        )


def one_hot(x, choices):
    return [int(x == c) for c in choices]


ATOM_LIST = [1,6,7,8,9,15,16,17,35,53]

def atom_features(a):
    return (
        one_hot(a.GetAtomicNum(), ATOM_LIST) + [a.GetAtomicNum() not in ATOM_LIST] +
        one_hot(a.GetDegree(), [0,1,2,3,4,5]) +
        one_hot(a.GetFormalCharge(), [-2,-1,0,1,2]) +
        one_hot(a.GetHybridization(), [
            Chem.rdchem.HybridizationType.SP,
            Chem.rdchem.HybridizationType.SP2,
            Chem.rdchem.HybridizationType.SP3,
            Chem.rdchem.HybridizationType.SP3D,
            Chem.rdchem.HybridizationType.SP3D2
        ]) +
        [int(a.GetIsAromatic())] +        # 🔥 dodaj
        [int(a.IsInRing())] +             # 🔥 dodaj
        [min(a.GetTotalNumHs(), 4) / 4.0]     # 🔥 dodaj
    )



BOND_DIM = 13   # 🔥 ustaw globalnie

# 🔥 bond features (~6 dim)
def bond_features(b):
    return (
        one_hot(b.GetBondTypeAsDouble(), [1, 2, 3, 1.5]) +   # 4
        [int(b.GetIsConjugated())] +                         # 1
        [int(b.IsInRing())] +                                # 1
        [int(b.GetIsAromatic())] +                           # 1
        [int(b.GetStereo() != Chem.rdchem.BondStereo.STEREONONE)] +  # 1
        [int(b.GetBondDir() != Chem.rdchem.BondDir.NONE)] +          # 🔥 NOWE
        one_hot(b.GetStereo(), [
            Chem.rdchem.BondStereo.STEREONONE,
            Chem.rdchem.BondStereo.STEREOANY,
            Chem.rdchem.BondStereo.STEREOZ,
            Chem.rdchem.BondStereo.STEREOE
        ])                                                   # 4
    )

# =========================
# GRAPH BUILDER
# =========================


def mol_to_graph(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        mol = Chem.MolFromSmiles("C")

    atom_f = [atom_features(a) for a in mol.GetAtoms()]

    edges = []
    bond_f = []

    for b in mol.GetBonds():
        i = b.GetBeginAtomIdx()
        j = b.GetEndAtomIdx()

        bf = bond_features(b)

        # 🔥 gwarancja rozmiaru
        if len(bf) != BOND_DIM:
            bf = bf + [0]*(BOND_DIM - len(bf))

        edges.append((i,j))
        bond_f.append(bf)

        edges.append((j,i))
        bond_f.append(bf)

    if len(edges) == 0:
        return (
            torch.tensor(atom_f, dtype=torch.float32),
            torch.zeros((2,0), dtype=torch.long),
            torch.zeros((0,BOND_DIM)),  # 🔥 KLUCZ
            torch.zeros((0,), dtype=torch.long)
        )

    edge_index = torch.tensor(edges, dtype=torch.long).T
    bond_f = torch.tensor(bond_f, dtype=torch.float32)

    edge_dict = {(i,j):k for k,(i,j) in enumerate(edges)}
    rev = torch.tensor([edge_dict[(j,i)] for (i,j) in edges])

    return torch.tensor(atom_f, dtype=torch.float32), edge_index, bond_f, rev


# =========================
# BATCH BUILDER
# =========================
def build_batch(smiles_list):
    xs, edges, bonds, revs, batch = [], [], [], [], []
    offset = 0
    e_offset = 0

    for i, s in enumerate(smiles_list):
        x, e, b, r = mol_to_graph(s)

        xs.append(x)
        edges.append(e + offset)
        bonds.append(b)
        revs.append(r + e_offset)

        #batch.append(torch.full((x.size(0),), i))
        batch.append(torch.full((x.size(0),), i, dtype=torch.long))

        offset += x.size(0)
        e_offset += e.size(1)

    return (
        torch.cat(xs),
        torch.cat(edges,1),
        torch.cat(bonds),
        torch.cat(revs),
        torch.cat(batch)
    )

# =========================
# D-MPNN (CHEMPROP STYLE)
# =========================
class MPNN(nn.Module):
    def __init__(self, atom_dim, bond_dim, hidden=300, depth=4):
        super().__init__()

        self.atom_emb = nn.Linear(atom_dim, hidden)

        self.W_i = nn.Linear(hidden + bond_dim, hidden)
        self.W_h = nn.Linear(hidden, hidden)
        self.W_o = nn.Linear(hidden + hidden, hidden)

        self.dropout = nn.Dropout(0.2)
        self.depth = depth

        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)
            else:
                nn.init.zeros_(p)

    def forward(self, atom, edge, bond, rev, batch):

        h = self.dropout(self.atom_emb(atom))   # 🔥 FIX 1

        src, dst = edge

        m = F.relu(self.W_i(torch.cat([h[src], bond], dim=1)))

        for _ in range(self.depth - 1):
            nei = torch.zeros_like(m)
            nei.index_add_(0, rev, m)

            m = nei - m

            # 🔥 FIX 2 (edge dropout)
            mask = (torch.rand(m.size(0), device=m.device) > 0.1).float().unsqueeze(1)
            m = m * mask

            m = self.dropout(m)
            m = F.relu(self.W_h(m))

        agg = torch.zeros_like(h)
        agg.index_add_(0, dst, m)

        h = h + self.dropout(F.relu(self.W_o(torch.cat([h, agg], dim=1))))

        z = torch.zeros(
        batch.max()+1,
        h.size(1),
        device=h.device,
        dtype=h.dtype   )
        z.index_add_(0, batch, h)

        return z




# =========================
# FULL MODEL (QW-MTL)
# =========================
class Model(nn.Module):
    def __init__(self, atom_dim, bond_dim, n_tasks=13, depth=4):
        super().__init__()

        self.mpnn = MPNN(atom_dim, bond_dim, hidden=300, depth=depth)

        # 🔥 FIX 3 (BatchNorm)
        #self.input_norm = nn.BatchNorm1d(300 + 208)
        self.input_norm = nn.LayerNorm(300 + 208)

        # 🔥 FIX 4 (MLP + dropout)
        self.shared = nn.Sequential(
            nn.Linear(300+208, 512),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(512, 256)
        )

        self.heads = nn.ModuleList([
            nn.Linear(256,1) for _ in range(n_tasks)
        ])

        self.log_beta = nn.Parameter(torch.ones(n_tasks))
        #self.log_beta = nn.Parameter(torch.zeros(n_tasks))

        # 🔥 FIX 5 (init heads + MLP)
        for m in self.shared:
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                nn.init.zeros_(m.bias)

        for head in self.heads:
            nn.init.xavier_uniform_(head.weight)
            nn.init.zeros_(head.bias)



    def forward(self, atom, edge, bond, rev, batch, x):

      device = x.device

      atom = atom.to(device)
      edge = edge.to(device)
      bond = bond.to(device).float()
      rev = rev.to(device)
      batch = batch.to(device).long()

      z = self.mpnn(atom, edge, bond, rev, batch)

      # 🔥 stabilizacja embeddingu
      z = z / (z.norm(dim=1, keepdim=True) + 1e-6)

      h = torch.cat([z, x], dim=1)

      # 🔥 BatchNorm + dropout
      h = self.input_norm(h)
      h = F.dropout(h, 0.1, training=self.training)

      h = self.shared(h)

      out = torch.cat([head(h) for head in self.heads], dim=1)

      return out




In [ ]:
pos_weight = []
for t in range(y.shape[1]):
    m = mask[train_idx, t] == 1
    y_t = y[train_idx, t][m]

    pos = y_t.sum()
    neg = len(y_t) - pos

    pw = neg / (pos + 1e-8)
    pw = np.clip(pw, 1.0, 20.0)

    pos_weight.append(pw)

pos_weight = torch.tensor(pos_weight, dtype=torch.float32)

# =========================
# LOSS (paper exact)
# =========================


def loss_fn(logits, y, mask, log_beta, pos_weight):
    """
    logits: (batch, tasks)
    y:      (batch, tasks)
    mask:   (batch, tasks)
    log_beta: (tasks,)
    pos_weight: (tasks,)
    """

    device = logits.device
    pos_weight = pos_weight.to(device).float()

    # --- BCE (paper) ---
    bce = F.binary_cross_entropy_with_logits(
        logits,
        y,
        reduction="none",
        pos_weight=pos_weight
    )

    # --- mask ---
    bce = bce * mask

    # --- per-task loss ---
    nt = mask.sum(dim=0)                                  # (tasks,)
    Lt = bce.sum(dim=0) / torch.clamp(nt, min=1.0)         # (tasks,)

    # --- proportions ---
    rt = nt / (nt.sum() + 1e-8)                            # (tasks,)

    # --- learnable weighting ---
    #beta = 0.5 + F.softplus(log_beta) * 0.5
    beta = F.softplus(log_beta)

    wt = rt ** beta                                        # (tasks,)

    # --- final ---
    loss = (wt * Lt).sum()

    return loss





In [ ]:
import os
import numpy as np
import torch
from sklearn.metrics import roc_auc_score
import seaborn as sns
import matplotlib.pyplot as plt

# =========================
# AUC
# =========================
def compute_auc(y_true, y_pred, mask):
    aucs = []
    for t in range(y_true.shape[1]):
        m = mask[:, t] == 1
        if m.sum() > 10:
            aucs.append(roc_auc_score(y_true[m, t], y_pred[m, t]))
        else:
            aucs.append(np.nan)
    return np.array(aucs)

# =========================
# EVAL
# =========================
def evaluate(model, loader, device):
    model.eval()

    Y, P, M = [], [], []

    with torch.no_grad():
        for atom, edge, bond, rev, batch_idx, X, y, m in loader:

            atom = atom.to(device)
            edge = edge.to(device)
            bond = bond.to(device)
            rev = rev.to(device)
            batch_idx = batch_idx.to(device)

            X = X.to(device)

            logits = model(atom, edge, bond, rev, batch_idx, X)
            probs = torch.sigmoid(logits).cpu().numpy()

            Y.append(y.numpy())
            P.append(probs)
            M.append(m.numpy())

    Y = np.concatenate(Y)
    P = np.concatenate(P)
    M = np.concatenate(M)

    return compute_auc(Y, P, M)


def save_auc_per_task(auc, split_name="val", save_dir="stats"):
    import os
    os.makedirs(save_dir, exist_ok=True)

    with open(f"{save_dir}/{split_name}_auc.txt", "w") as f:
        for i, a in enumerate(auc):
            f.write(f"Task {i:02d}: {a:.4f}\n")

    print(f"AUC saved → {save_dir}/{split_name}_auc.txt")



from sklearn.metrics import roc_curve

def plot_roc_per_task(y, pred, mask, split_name="val", save_dir="plots_roc"):
    import os
    os.makedirs(save_dir, exist_ok=True)

    for t in range(y.shape[1]):
        m = mask[:, t] == 1
        if m.sum() < 20:
            continue

        y_t = y[m, t]
        p_t = pred[m, t]

        fpr, tpr, _ = roc_curve(y_t, p_t)

        plt.figure()
        plt.plot(fpr, tpr)
        plt.xlabel("FPR")
        plt.ylabel("TPR")
        plt.title(f"ROC Task {t:02d}")

        plt.savefig(f"{save_dir}/{split_name}_task_{t:02d}.png")
        plt.close()



def full_evaluation(model, loader, device, split_name="val"):
    model.eval()

    Y, P, M = [], [], []

    with torch.no_grad():
        for atom, edge, bond, rev, batch_idx, X, y, m in loader:

            atom = atom.to(device)
            edge = edge.to(device)
            bond = bond.to(device)
            rev = rev.to(device)
            batch_idx = batch_idx.to(device)

            X = X.to(device)

            logits = model(atom, edge, bond, rev, batch_idx, X)
            probs = torch.sigmoid(logits).cpu().numpy()

            Y.append(y.numpy())
            P.append(probs)
            M.append(m.numpy())

    Y = np.concatenate(Y)
    P = np.concatenate(P)
    M = np.concatenate(M)

    auc = compute_auc(Y, P, M)

    save_auc_per_task(auc, split_name)
    plot_roc_per_task(Y, P, M, split_name)
    plot_prediction_histograms(Y, P, M, split_name)

    print(f"{split_name} mean AUC:", np.nanmean(auc))

    return auc


def plot_prediction_histograms(y, pred, mask, split_name="val", save_dir="plots_hist"):
    import os
    os.makedirs(save_dir, exist_ok=True)

    for t in range(y.shape[1]):
        m = mask[:, t] == 1
        if m.sum() < 20:
            continue

        y_t = y[m, t]
        p_t = pred[m, t]

        plt.figure()
        plt.hist(p_t[y_t == 0], bins=50, alpha=0.5, label="neg")
        plt.hist(p_t[y_t == 1], bins=50, alpha=0.5, label="pos")
        plt.legend()
        plt.title(f"Pred dist Task {t:02d}")

        plt.savefig(f"{save_dir}/{split_name}_task_{t:02d}.png")
        plt.close()


def task_statistics(y, mask, split_name="dataset", save_dir="stats"):
    import os
    os.makedirs(save_dir, exist_ok=True)

    stats = []

    for t in range(y.shape[1]):
        m = mask[:, t] == 1
        y_t = y[m, t]

        if len(y_t) == 0:
            continue

        pos = y_t.sum()
        neg = len(y_t) - pos
        ratio = pos / (len(y_t) + 1e-8)

        stats.append((t, len(y_t), int(pos), int(neg), ratio))

    # save txt
    with open(f"{save_dir}/{split_name}_stats.txt", "w") as f:
        for t, n, pos, neg, ratio in stats:
            f.write(f"Task {t:02d} | n={n} | pos={pos} | neg={neg} | ratio={ratio:.3f}\n")

    print(f"Saved stats → {save_dir}/{split_name}_stats.txt")




# =========================
# PLOT
# =========================
def plot_heatmap(auc, title, save_path=None):
    plt.figure(figsize=(12,2))
    sns.heatmap(auc.reshape(1,-1), annot=True, fmt=".3f", cmap="viridis")
    plt.title(title)

    if save_path:
        plt.savefig(save_path, bbox_inches="tight")
    plt.close()

# =========================
# TRAIN
# =========================

def train_model(model, train_loader, val_loader, device, epochs=40):

    torch.backends.cudnn.benchmark = True

    model.to(device)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=3e-4,
        weight_decay=1e-5
    )

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode='max',
        factor=0.7,
        patience=4,
        min_lr=1e-6
    )

    scaler = torch.amp.GradScaler("cuda")

    pos_weight_local = pos_weight.to(device)

    best_auc = -1
    history = []
    patience = 6
    no_improve = 0

    os.makedirs("checkpoints", exist_ok=True)
    os.makedirs("plots", exist_ok=True)

    for epoch in range(epochs):
        model.train()
        total_loss = 0

        for atom, edge, bond, rev, batch_idx, X, y, m in train_loader:

            atom = atom.to(device)
            edge = edge.to(device)
            bond = bond.to(device)
            rev = rev.to(device)
            batch_idx = batch_idx.to(device)

            X = X.to(device)
            y = y.to(device)
            m = m.to(device)

            optimizer.zero_grad()

            with torch.amp.autocast("cuda"):
                logits = model(atom, edge, bond, rev, batch_idx, X)

                loss = loss_fn(logits, y, m, model.log_beta, pos_weight_local)
                loss = loss + 1e-5 * (model.log_beta ** 2).sum()

            scaler.scale(loss).backward()

            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)

            scaler.step(optimizer)
            scaler.update()

            model.log_beta.data.clamp_(-3, 3)

            total_loss += loss.item() * X.size(0)

        total_loss /= len(train_loader.dataset)

        # ===== EVAL =====
        val_auc = evaluate(model, val_loader, device)
        mean_auc = np.nanmean(val_auc)


        scheduler.step(mean_auc)

        history.append(mean_auc)

        lr = optimizer.param_groups[0]["lr"]

        print(f"Epoch {epoch:02d} | Loss {total_loss:.4f} | Val AUC {mean_auc:.4f} | LR {lr:.6f}")



        print(f"\nEpoch {epoch:02d}")
        print(f"Loss: {total_loss:.4f}")
        print(f"Val AUC: {mean_auc:.4f}")
        print(f"LR: {lr:.6e}")

        # 🔥 per-task AUC
        print("AUC per task:", np.round(val_auc, 3))

        # 🔥 beta
        print("beta:", torch.nn.functional.softplus(model.log_beta).detach().cpu().numpy())


        if epoch % 5 == 0:
            print("beta:", torch.nn.functional.softplus(model.log_beta))

        # ===== SAVE BEST =====
        if mean_auc > best_auc:
            best_auc = mean_auc
            no_improve = 0

            torch.save(model.state_dict(), "checkpoints/best.pt")

            plot_heatmap(val_auc, f"Val AUC epoch {epoch}", f"plots/val_auc_epoch_{epoch}.png")
        else:
            no_improve += 1

        # ===== EARLY STOP =====
        if no_improve >= patience:
            print("Early stopping")
            break

    return history



In [50]:
import os
import shutil

# ===== CLEAN OLD RUN FILES =====

def clean_run_dirs():
    folders = [
        "plots",
        "plots_hist",
        "plots_roc",
        "stats"
    ]

    files = [
        "history.npy",
        "run_stats.txt"
    ]

    # usuń foldery
    for f in folders:
        if os.path.exists(f):
            shutil.rmtree(f)

    # usuń pliki
    for f in files:
        if os.path.exists(f):
            os.remove(f)

    print("✔ cleaned previous run artifacts")


# ===== CALL BEFORE MAIN =====
clean_run_dirs()

✔ cleaned previous run artifacts


In [51]:
import torch
import numpy as np
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt


from rdkit import RDLogger
RDLogger.DisableLog('rdApp.*')



def main():


    # =========================
    # DEVICE
    # =========================
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Device:", device)

    # =========================
    # DATASET
    # =========================
    train_ds = MolDataset(smiles_list, X, y, mask, train_idx)
    val_ds   = MolDataset(smiles_list, X, y, mask, val_idx)
    test_ds  = MolDataset(smiles_list, X, y, mask, test_idx)

    train_loader = DataLoader(
        train_ds,
        batch_size=1024,
        shuffle=True,
        collate_fn=collate_fn,
        num_workers=0,
        pin_memory=True
    )

    val_loader = DataLoader(
        val_ds,
        batch_size=1024,
        collate_fn=collate_fn
    )

    test_loader = DataLoader(
        test_ds,
        batch_size=1024,
        collate_fn=collate_fn
    )

    # =========================
    # MODEL INIT
    # =========================
    a, e, b, r = mol_to_graph(smiles_list[0])

    atom_dim = a.shape[1]
    bond_dim = b.shape[1]

    print("atom_dim:", atom_dim, "bond_dim:", bond_dim)

    model = Model(atom_dim=atom_dim, bond_dim=bond_dim, n_tasks=y.shape[1])



    print("\n=== MODEL INFO ===")
    print("Params:", sum(p.numel() for p in model.parameters()) / 1e6, "M")
    print("Tasks:", y.shape[1])

    # =========================
    # TRAIN
    # =========================
    history = train_model(
        model,
        train_loader,
        val_loader,
        device,
        epochs=60
    )

    # =========================
    # LOAD BEST
    # =========================
    model.load_state_dict(torch.load("checkpoints/best.pt", map_location=device))
    model.to(device)

    # =========================
    # FULL EVAL
    # =========================
    val_auc = full_evaluation(model, val_loader, device, "val")
    test_auc = full_evaluation(model, test_loader, device, "test")


    print("\n=== FINAL RESULTS ===")
    print("VAL AUC:", np.nanmean(val_auc))
    print("TEST AUC:", np.nanmean(test_auc))

    plot_heatmap(test_auc, "Test AUC", "plots/test_auc.png")

    # =========================
    # SAVE HISTORY
    # =========================
    np.save("run/history.npy", np.array(history))

    # =========================
    # PLOT LEARNING CURVE
    # =========================
    plt.figure()
    plt.plot(history)
    plt.xlabel("Epoch")
    plt.ylabel("Val AUC")
    plt.title("Learning Curve")
    plt.savefig("plots/learning_curve.png")
    plt.close()


# =========================
# RUN
# =========================
if __name__ == "__main__":
    main()

Device: cuda
atom_dim: 30 bond_dim: 13

=== MODEL INFO ===
Params: 1.033062 M
Tasks: 13
Epoch 00 | Loss 0.4018 | Val AUC 0.7363 | LR 0.000300

Epoch 00
Loss: 0.4018
Val AUC: 0.7363
LR: 3.000000e-04
AUC per task: [0.776 0.841 0.64  0.684 0.841 0.539 0.827 0.592 0.835 0.781 0.78  0.602
 0.833]
beta: [1.3226346 1.3223268 1.322028  1.3225601 1.3225179 1.3216771 1.3226854
 1.3221885 1.3226488 1.3223052 1.3226678 1.3213388 1.3222747]
beta: tensor([1.3226, 1.3223, 1.3220, 1.3226, 1.3225, 1.3217, 1.3227, 1.3222, 1.3226,
        1.3223, 1.3227, 1.3213, 1.3223], device='cuda:0',
       grad_fn=<SoftplusBackward0>)
Epoch 01 | Loss 0.3439 | Val AUC 0.7952 | LR 0.000300

Epoch 01
Loss: 0.3439
Val AUC: 0.7952
LR: 3.000000e-04
AUC per task: [0.79  0.878 0.702 0.689 0.861 0.762 0.85  0.597 0.846 0.862 0.802 0.825
 0.874]
beta: [1.331744  1.3303905 1.3301424 1.331056  1.3314024 1.3299885 1.3318377
 1.3311253 1.3315777 1.3307157 1.331658  1.3272558 1.3291129]
Epoch 02 | Loss 0.3235 | Val AUC 0.8172 | LR